# Anti-UAV YOLO26s Training — Fast Validation Run
**Model:** yolo26s | **Hardware:** Colab T4 | **Dataset:** 30% of merged (Bird/Drone/UAV)

**Estimated time:** 3-4 hours | **Early stopping:** patience=20

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Runtime → Run all

In [ ]:
# Cleanup any previous cancelled run
import shutil, os
for old in ['/content/runs/anti_uav_run2_yolo26m', '/content/runs/anti_uav_run1_yolo26s']:
    if os.path.exists(old):
        shutil.rmtree(old)
        print(f'Deleted: {old}')
print('Clean')


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())


In [ ]:
# Find and extract dataset from Drive
import tarfile, os, yaml

tar_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f == 'backup_merged_dataset.tar.gz':
            tar_path = os.path.join(root, f)
            break
    if tar_path:
        break

if tar_path is None:
    raise FileNotFoundError('backup_merged_dataset.tar.gz not found in Google Drive.')

print(f'Found: {tar_path}')
extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)
print('Extracting (~2 min)...')
with tarfile.open(tar_path) as tf:
    tf.extractall(extract_dir)

data_yaml = os.path.join(extract_dir, 'merged_dataset', 'data.yaml')
base = os.path.join(extract_dir, 'merged_dataset')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'val', 'images')
cfg['test'] = os.path.join(base, 'test', 'images')
with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f)

# Count images
for split in ['train', 'val', 'test']:
    n = len(os.listdir(cfg[split]))
    print(f'  {split}: {n} images (using 30% = ~{int(n*0.3)})')


In [ ]:
# Install/upgrade ultralytics and restart runtime
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics>=8.4.0'], check=True)
import ultralytics
print(f'ultralytics {ultralytics.__version__}')
assert tuple(int(x) for x in ultralytics.__version__.split('.')[:2]) >= (8, 4), \
    f'Need ultralytics>=8.4.0, got {ultralytics.__version__}. Restart runtime and re-run.'
print('Version OK — YOLO26 supported')


In [ ]:
# Run training — yolo26s fast validation
# Background thread copies checkpoints to Drive every 10 epochs
import threading, time, shutil, os
from ultralytics import YOLO

DRIVE_BACKUP = '/content/drive/MyDrive/anti_uav_checkpoints'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
RUNS_DIR = '/content/runs/anti_uav_run1_yolo26s/weights'
stop_backup = threading.Event()

def backup_to_drive():
    while not stop_backup.is_set():
        time.sleep(600)  # every 10 minutes
        if os.path.exists(RUNS_DIR):
            for f in os.listdir(RUNS_DIR):
                src = os.path.join(RUNS_DIR, f)
                dst = os.path.join(DRIVE_BACKUP, f)
                try:
                    shutil.copy2(src, dst)
                    print(f'[backup] Saved {f} to Drive')
                except Exception as e:
                    print(f'[backup] Failed {f}: {e}')

backup_thread = threading.Thread(target=backup_to_drive, daemon=True)
backup_thread.start()
print('Checkpoint backup thread started (saves to Drive every 10 min)')

model = YOLO('yolo26s.pt')
results = model.train(
    data=data_yaml,
    imgsz=640,
    batch=32,
    epochs=100,
    patience=20,
    fraction=0.3,
    optimizer='MuSGD',
    lr0=0.01,
    weight_decay=0.0005,
    amp=True,
    device='0',
    save_period=10,
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=20.0,
    translate=0.15,
    scale=0.8,
    flipud=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='anti_uav_run1_yolo26s',
)
stop_backup.set()
print(f'Training complete — results: {results.save_dir}')


In [ ]:
# Archive full run and save to Drive
import zipfile, os, shutil
runs_dir = '/content/runs/anti_uav_run1_yolo26s'
archive_local = '/content/anti_uav_run1_yolo26s_full.zip'
archive_drive = '/content/drive/MyDrive/anti_uav_run1_yolo26s_full.zip'

print('Archiving full run directory...')
with zipfile.ZipFile(archive_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(runs_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, '/content')
            zf.write(filepath, arcname)
            print(f'  {arcname} ({os.path.getsize(filepath)/1e6:.1f} MB)')

shutil.copy2(archive_local, archive_drive)
print(f'Saved to Drive: {archive_drive} ({os.path.getsize(archive_drive)/1e6:.1f} MB)')
print('Done')
